# Does the CN effect depend on who posted the tweet?

**Data used:** extracted features (variables we generated with Gemini / Apify / XPOZ)

**Short answer:** No. Followers, account age, bio, avatar, bot-likelihood — none of it mattered.



# User Features Analysis

Loads `user_features.csv` (built by `poster_features_build.ipynb`) and tests whether poster characteristics moderate the CN suppression effect.

**For each feature, three complementary analyses:**

| Analysis | Question | Method |
|---|---|---|
| Balance check | Are user features balanced across Control / Treatment? | Distribution comparison (not assumed balanced) |
| Stratified suppression | Does the CN effect differ by feature tier? | Control vs Treatment within each quartile/category → forest plot |
| Interaction regression | Does the feature significantly moderate the effect? | OLS: `growth ~ group × feature + log_baseline` |

**Sign convention (consistent with main analysis):**
`rank_biserial_r(trt, ctrl)` → r > 0 = Control grew more = **suppression**; r < 0 = Treatment grew more = **anti-suppression**.

---
## 0. Configuration

In [ ]:
import os

# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then take the field-experiment folder inside it.
_root = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_root, "requirements.txt")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
BASE_DIR = os.path.join(_root, "field-experiment")
if not os.path.isdir(os.path.join(BASE_DIR, "data")):
    raise RuntimeError(
        "Could not locate the field-experiment folder from " + os.getcwd() +
        ". Run this notebook from inside the cloned repository."
    )
DATA_DIR    = os.path.join(BASE_DIR, "data")
OUT_DIR     = os.path.join(BASE_DIR, "outputs", "poster_features")
FEATURES_CSV = os.path.join(OUT_DIR, "user_features.csv")

RANDOM_SEED  = 42
N_BOOTSTRAP  = 5000
MAIN_WINDOW  = 13   # primary growth window (days)
METRICS      = ["Views", "Likes", "Shares"]

print("Features CSV:", FEATURES_CSV)
print("Exists:", os.path.exists(FEATURES_CSV))

---
## 1. Imports & Helpers

In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

# ── Sign-correct rank-biserial r (trt first, ctrl second) ─────────────────────
# r > 0 → ctrl > trt → suppression
# r < 0 → trt > ctrl → anti-suppression / backfire
def rank_biserial_r(trt, ctrl):
    nx, ny = len(trt), len(ctrl)
    U, _   = stats.mannwhitneyu(trt, ctrl, alternative="two-sided")
    return 1 - (2 * U) / (nx * ny)


def bootstrap_ci(trt, ctrl, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    boots = [
        rank_biserial_r(
            rng.choice(trt,  size=len(trt),  replace=True),
            rng.choice(ctrl, size=len(ctrl), replace=True),
        )
        for _ in range(n_boot)
    ]
    return np.percentile(boots, 2.5), np.percentile(boots, 97.5)


def compute_effect(trt_vals, ctrl_vals):
    """Returns dict with r, ci_lo, ci_hi, p, n_trt, n_ctrl."""
    trt_vals  = np.array(trt_vals, dtype=float)
    ctrl_vals = np.array(ctrl_vals, dtype=float)
    trt_vals  = trt_vals[~np.isnan(trt_vals)]
    ctrl_vals = ctrl_vals[~np.isnan(ctrl_vals)]
    if len(trt_vals) < 5 or len(ctrl_vals) < 5:
        return None
    _, p    = stats.mannwhitneyu(trt_vals, ctrl_vals, alternative="two-sided")
    r       = rank_biserial_r(trt_vals, ctrl_vals)
    lo, hi  = bootstrap_ci(trt_vals, ctrl_vals)
    return {"r": r, "ci_lo": lo, "ci_hi": hi, "p": p,
            "n_trt": len(trt_vals), "n_ctrl": len(ctrl_vals),
            "sig": "*" if p < 0.05 else ("." if p < 0.10 else "")}


print("Helpers loaded.")

---
## 2. Load Data

In [ ]:
# ── User features ──────────────────────────────────────────────────────────────
user_features = pd.read_csv(FEATURES_CSV)
print(f"User features: {user_features.shape}")

# ── Rebuild complete_df from raw data (same pipeline as main notebook) ──────────
control    = pd.read_excel(os.path.join(DATA_DIR, "Control_Group.xlsx"))
treatment  = pd.read_excel(os.path.join(DATA_DIR, "Treatment_Group.xlsx"))
monitoring = pd.read_excel(os.path.join(DATA_DIR, "Tweet Monitoring.xlsx"))

def parse_mixed_date(v):
    if pd.isna(v): return pd.NaT
    if isinstance(v, pd.Timestamp): return v
    for fmt in ["%Y-%m-%d %H:%M:%S","%Y-%m-%d","%d/%m/%Y","%m/%d/%Y"]:
        try:
            dt = pd.to_datetime(str(v).strip(), format=fmt)
            if 1 <= dt.month <= 12: return dt
        except: pass
    try: return pd.to_datetime(str(v).strip(), dayfirst=True)
    except: return pd.NaT

monitoring["Sample_Date"]          = monitoring["Sample_Date"].apply(parse_mixed_date)
monitoring["Start_Date (Creation)"] = monitoring["Start_Date (Creation)"].apply(parse_mixed_date)
monitoring["Day"] = (monitoring["Sample_Date"] - monitoring["Start_Date (Creation)"]).dt.days

ctrl_meta = control[["URL","User","Type","Narrative","Group","Views","Likes","Comments","Shares"]].copy()
trt_meta  = treatment[["URL","User","Type","Narrative","Group","Views","Likes","Comments","Shares"]].copy()
tweet_meta = pd.concat([ctrl_meta, trt_meta], ignore_index=True)

pivoted = monitoring.pivot_table(
    index="URL", columns="Day",
    values=["Views","Likes","Comments","Shares"], aggfunc="first"
)
pivoted.columns = [f"{m}_Day{d}" for m, d in pivoted.columns]
pivoted = pivoted.reset_index()

mon_days = monitoring.groupby("URL")["Day"].nunique().reset_index()
mon_days.columns = ["URL", "Num_Days"]

analysis_df = tweet_meta.merge(pivoted, on="URL", how="left").merge(mon_days, on="URL", how="left")
analysis_df["Complete_Monitoring"] = analysis_df["Num_Days"] >= 14

for metric in ["Likes","Shares","Views","Comments"]:
    for window in [1, 3, 7, 13]:
        d0 = f"{metric}_Day0"; dN = f"{metric}_Day{window}"
        if d0 in analysis_df.columns and dN in analysis_df.columns:
            analysis_df[f"{metric}_Growth_{window}d"] = (
                (analysis_df[dN] - analysis_df[d0]) / (analysis_df[d0] + 1) * 100
            )

complete_df = analysis_df[analysis_df["Complete_Monitoring"]].copy()
print(f"complete_df: {len(complete_df)} tweets  (ctrl={len(complete_df[complete_df.Group=='Control'])}, trt={len(complete_df[complete_df.Group=='Treatment'])})")

In [ ]:
# ── Merge user features into tweet-level dataframe ─────────────────────────────
# Use handle (extracted from tweet URL) as the join key — display names are not unique.
complete_df["tweet_handle"] = (
    complete_df["URL"]
    .str.extract(r"x\.com/([^/]+)/status", expand=False)
    .str.strip().str.lower()
)

df = complete_df.merge(
    user_features.drop(columns=["description_short", "avatar", "user_name"], errors="ignore"),
    left_on="tweet_handle", right_on="handle", how="left"
)

ctrl_df = df[df["Group"] == "Control"]
trt_df  = df[df["Group"] == "Treatment"]

print(f"Merged df: {len(df)} rows")
print(f"User feature coverage: {df['log_followers'].notna().mean():.1%}")
print(f"  (unmatched: {df['log_followers'].isna().sum()} tweets — no EU record for those authors)")

In [ ]:
# ── Fix 1: normalize xpoz_verified to clean bool ──────────────────────────────
# Field can be True/False, 1/0, or None depending on source
if "xpoz_verified" in df.columns:
    df["xpoz_verified"] = df["xpoz_verified"].map(
        {True: True, False: False, 1: True, 0: False, "True": True, "False": False}
    )  # leaves None/NaN as NaN — excluded from all analyses automatically

# ── Fix 2: coverage-based feature filter ──────────────────────────────────────
MIN_COVERAGE = 0.20   # skip features with <20% non-null values

ALL_NUMERIC = [
    "log_followers", "log_following", "ff_ratio", "account_age_days",
    "bio_stance_score", "xpoz_inauthentic_prob", "xpoz_avg_tweets_day",
]
ALL_CAT = ["bio_account_type", "avatar_type", "xpoz_inauthentic_type"]

print(f"{'Feature':<30} {'Coverage':>10}  {'Status':>8}")
print("-" * 55)

VALID_NUMERIC, VALID_CAT = [], []

for feat in ALL_NUMERIC:
    cov = df[feat].notna().mean() if feat in df.columns else 0.0
    status = "OK" if cov >= MIN_COVERAGE else "SKIP"
    print(f"{feat:<30} {cov:>9.1%}  {status:>8}")
    if cov >= MIN_COVERAGE:
        VALID_NUMERIC.append(feat)

for feat in ALL_CAT:
    cov = df[feat].notna().mean() if feat in df.columns else 0.0
    status = "OK" if cov >= MIN_COVERAGE else "SKIP"
    print(f"{feat:<30} {cov:>9.1%}  {status:>8}")
    if cov >= MIN_COVERAGE:
        VALID_CAT.append(feat)

ver_cov = df["xpoz_verified"].notna().mean() if "xpoz_verified" in df.columns else 0.0
ver_ok  = ver_cov >= MIN_COVERAGE
print(f"{'xpoz_verified':<30} {ver_cov:>9.1%}  {'OK' if ver_ok else 'SKIP':>8}")

print(f"\nValid numeric   : {VALID_NUMERIC}")
print(f"Valid categorical: {VALID_CAT}")

# ── Reassign group subsets so they include any new columns added above ─────────
ctrl_df = df[df["Group"] == "Control"]
trt_df  = df[df["Group"] == "Treatment"]

---
## 3. Balance Check

Since randomization was not stratified by user-level features, we empirically test whether they are balanced across groups. An imbalance would mean some confounding; significant imbalances should be noted as limitations.

In [ ]:
print("=" * 75)
print("BALANCE CHECK — User Features Across Control vs Treatment")
print("(Mann-Whitney U; p < 0.05 = imbalance to flag)")
print("=" * 75)
print(f"{'Feature':<28} {'Ctrl Med':>10} {'Trt Med':>10} {'p-value':>10} {'flag':>6}")
print("-" * 68)

for feat in VALID_NUMERIC:
    c = ctrl_df[feat].dropna().values
    t = trt_df[feat].dropna().values
    if len(c) < 5 or len(t) < 5:
        continue
    _, p = stats.mannwhitneyu(c, t, alternative="two-sided")
    flag = "⚠" if p < 0.05 else ""
    print(f"{feat:<28} {np.median(c):>10.2f} {np.median(t):>10.2f} {p:>10.4f} {flag:>6}")

In [ ]:
# ── Balance: categorical features (coverage-filtered) ─────────────────────────
cat_to_check = VALID_CAT + (["xpoz_verified"] if ver_ok else [])

print("\n=== CATEGORICAL FEATURE BALANCE ===")
for feat in cat_to_check:
    if feat not in df.columns:
        continue
    # dropna=True is default in crosstab — NaN rows excluded automatically
    tbl = pd.crosstab(df[feat], df["Group"], normalize="columns").round(3)
    print(f"\n{feat}  (n={df[feat].notna().sum()} non-null):")
    print(tbl.to_string())

---
## 4. Stratified Suppression Analysis

For each feature, we compare Control vs Treatment growth within each tier and plot effect sizes.

In [ ]:
# ── Reusable forest plot function ──────────────────────────────────────────────
COLORS = {"Views": "steelblue", "Likes": "seagreen", "Shares": "firebrick"}

def forest_plot(results_df, title, ax=None, show_legend=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, max(4, len(results_df) * 0.5)))

    labels  = results_df["label"].unique()
    offsets = {m: (i - 1) * 0.15 for i, m in enumerate(METRICS)}

    for metric in METRICS:
        sub = results_df[results_df["metric"] == metric]
        y   = [list(labels).index(l) + offsets[metric] for l in sub["label"]]
        ax.errorbar(sub["r"], y,
                    xerr=[sub["r"] - sub["ci_lo"], sub["ci_hi"] - sub["r"]],
                    fmt="o", color=COLORS[metric], label=metric,
                    markersize=7, capsize=4, linewidth=1.5)
        sig_mask = sub["sig"] == "*"
        if sig_mask.any():
            sig_y = [list(labels).index(l) + offsets[metric] for l in sub.loc[sig_mask, "label"]]
            ax.scatter(sub.loc[sig_mask, "r"], sig_y, s=60, marker="*",
                       color=COLORS[metric], zorder=5)

    ax.axvline(0, color="black", linewidth=1.2, linestyle="--", alpha=0.6)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("r (trt vs ctrl)  |  r > 0 = suppression, r < 0 = anti-suppression")
    ax.set_title(title, fontweight="bold")
    if show_legend:
        ax.legend(loc="lower right", fontsize=9)
    return ax


def stratified_effects(df, feature_col, quartile=True, n_quantiles=4):
    """Compute Control vs Treatment effect sizes per tier of feature_col.
    NaN values are always excluded. Quartile labels adapt if duplicate bin
    edges reduce the actual number of bins (e.g. integer scores like 1–5).
    """
    results = []
    df = df.copy()
    df = df[df[feature_col].notna()]

    if quartile:
        # labels=False → integer codes; avoids label-count mismatch when
        # duplicates="drop" silently reduces the number of bins
        binned = pd.qcut(df[feature_col], q=n_quantiles,
                         labels=False, duplicates="drop")
        n_actual = int(binned.max()) + 1
        df["_tier"] = binned.map({i: f"Q{i+1}" for i in range(n_actual)})
    else:
        df["_tier"] = df[feature_col].astype(str)

    for tier, tier_df in df.groupby("_tier", observed=True):
        ctrl_tier = tier_df[tier_df["Group"] == "Control"]
        trt_tier  = tier_df[tier_df["Group"] == "Treatment"]
        for metric in METRICS:
            col = f"{metric}_Growth_{MAIN_WINDOW}d"
            eff = compute_effect(trt_tier[col].dropna(), ctrl_tier[col].dropna())
            if eff:
                results.append({"label": str(tier), "metric": metric, **eff})
    return pd.DataFrame(results)


print("Forest plot helper ready.")

In [ ]:
# ── 4.1 Follower count (quartiles) ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
res = stratified_effects(df, "log_followers", quartile=True)
forest_plot(res, "CN Effect by Follower Count Quartile (Q1=fewest, Q4=most)", ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_followers_quartile.png"), bbox_inches="tight")
plt.show()

# Dose-response: is r monotone Q1→Q4?
for metric in METRICS:
    sub = res[res["metric"] == metric].set_index("label")["r"]
    trend = "monotone ↑" if list(sub) == sorted(sub) else "monotone ↓" if list(sub) == sorted(sub, reverse=True) else "non-monotone"
    print(f"{metric}: {[f'{v:+.3f}' for v in sub.values]}  → {trend}")

In [ ]:
# ── 4.2 Verified status ────────────────────────────────────────────────────────
if "xpoz_verified" in df.columns and df["xpoz_verified"].notna().any():
    df["_verified_str"] = df["xpoz_verified"].map({True: "Verified", False: "Not verified",
                                                    1: "Verified", 0: "Not verified"})
    # NaNs are left as NaN so stratified_effects excludes them automatically
    res_v = stratified_effects(df, "_verified_str", quartile=False)
    fig, ax = plt.subplots(figsize=(10, 4))
    forest_plot(res_v, "CN Effect: Verified vs Non-Verified Accounts", ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig_verified.png"), bbox_inches="tight")
    plt.show()
else:
    print("xpoz_verified not available — skipping.")

In [ ]:
# ── 4.3 Inauthenticity score (quartiles) ──────────────────────────────────────
if "xpoz_inauthentic_prob" in df.columns and df["xpoz_inauthentic_prob"].notna().any():
    res_bot = stratified_effects(df, "xpoz_inauthentic_prob", quartile=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    forest_plot(res_bot, "CN Effect by Bot Probability Quartile (Q1=most human, Q4=most bot-like)", ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig_inauthentic_quartile.png"), bbox_inches="tight")
    plt.show()

In [ ]:
# ── 4.4 Inauthentic type (categorical) ────────────────────────────────────────
if "xpoz_inauthentic_type" in df.columns:
    df["_inauthtype"] = df["xpoz_inauthentic_type"].fillna("Authentic / unknown")
    res_type = stratified_effects(df, "_inauthtype", quartile=False)
    if len(res_type) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(res_type["label"].unique()) * 0.8)))
        forest_plot(res_type, "CN Effect by Account Inauthenticity Type", ax=ax)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "fig_inauthentic_type.png"), bbox_inches="tight")
        plt.show()

In [ ]:
# ── 4.5 Account age (quartiles) ───────────────────────────────────────────────
res_age = stratified_effects(df, "account_age_days", quartile=True)
fig, ax = plt.subplots(figsize=(10, 5))
forest_plot(res_age, "CN Effect by Account Age Quartile (Q1=newest, Q4=oldest)", ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_account_age.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ── 4.6 Bio stance explicitness (quartiles) ───────────────────────────────────
if "bio_stance_score" in df.columns and df["bio_stance_score"].notna().any():
    res_stance = stratified_effects(df, "bio_stance_score", quartile=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    forest_plot(res_stance, "CN Effect by Pro-Russian Stance Explicitness (Q1=covert, Q4=overt)", ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig_bio_stance.png"), bbox_inches="tight")
    plt.show()

In [ ]:
# ── 4.7 Bio account type (categorical) ────────────────────────────────────────
if "bio_account_type" in df.columns and df["bio_account_type"].notna().any():
    res_actype = stratified_effects(df, "bio_account_type", quartile=False)
    if len(res_actype) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(res_actype["label"].unique()) * 0.8)))
        forest_plot(res_actype, "CN Effect by Account Type (from bio)", ax=ax)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "fig_bio_account_type.png"), bbox_inches="tight")
        plt.show()

In [ ]:
# ── 4.8 Avatar type (categorical) ─────────────────────────────────────────────
if "avatar_type" in df.columns and df["avatar_type"].notna().any():
    res_avtype = stratified_effects(df, "avatar_type", quartile=False)
    if len(res_avtype) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(res_avtype["label"].unique()) * 0.8)))
        forest_plot(res_avtype, "CN Effect by Avatar Type", ax=ax)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "fig_avatar_type.png"), bbox_inches="tight")
        plt.show()

In [ ]:
# ── 4.9 National symbol in avatar (binary) ────────────────────────────────────
if "avatar_national_symbol" in df.columns and df["avatar_national_symbol"].notna().any():
    df["_nat_sym"] = df["avatar_national_symbol"].map(
        {True: "Has symbol", False: "No symbol", 1: "Has symbol", 0: "No symbol"}
    ).fillna("Unknown")
    res_nat = stratified_effects(df, "_nat_sym", quartile=False)
    if len(res_nat) > 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        forest_plot(res_nat, "CN Effect: Avatar Has National/Political Symbol", ax=ax)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "fig_national_symbol.png"), bbox_inches="tight")
        plt.show()

---
## 5. Interaction Regression

For each continuous feature, tests whether it significantly moderates the CN effect using an interaction term.

`growth_13d ~ is_treatment * feature + log_baseline`

A significant `is_treatment:feature` coefficient means the feature changes how much CNs suppress growth.

In [ ]:
df["is_treatment"] = (df["Group"] == "Treatment").astype(int)

# Only features that passed the coverage filter
INTERACTION_FEATURES = [
    (f, f.replace("log_", "").replace("_", " ").replace("xpoz ", "").title())
    for f in VALID_NUMERIC
]

interaction_results = []

print("=" * 90)
print("INTERACTION REGRESSION: growth_13d ~ is_treatment * feature + log_baseline")
print("Coefficient on is_treatment:feature = moderation effect")
print("=" * 90)

for metric in METRICS:
    growth_col   = f"{metric}_Growth_{MAIN_WINDOW}d"
    baseline_col = f"{metric}_Day0"
    print(f"\n--- {metric} ---")

    for feat_col, feat_label in INTERACTION_FEATURES:
        if feat_col not in df.columns:
            continue
        sub = df[["is_treatment", feat_col, growth_col, baseline_col]].dropna().copy()
        if len(sub) < 30:
            print(f"  {feat_label:<35} SKIP (n={len(sub)} < 30)")
            continue
        sub["feat_std"] = (sub[feat_col] - sub[feat_col].mean()) / sub[feat_col].std()
        sub["log_base"] = np.log1p(sub[baseline_col])
        p99 = sub[growth_col].quantile(0.99)
        sub["growth_w"] = sub[growth_col].clip(upper=p99)

        try:
            model = smf.ols("growth_w ~ is_treatment * feat_std + log_base", data=sub).fit()
            coef = model.params.get("is_treatment:feat_std", np.nan)
            pval = model.pvalues.get("is_treatment:feat_std", np.nan)
            sig  = "*" if pval < 0.05 else ("." if pval < 0.10 else "")
            print(f"  {feat_label:<35} coef={coef:+.2f}  p={pval:.4f}  {sig}")
            interaction_results.append({"metric": metric, "feature": feat_label,
                                         "coef": coef, "p": pval, "sig": sig})
        except Exception as e:
            print(f"  {feat_label:<35} ERROR: {e}")

int_df = pd.DataFrame(interaction_results)
print("\n\nSignificant interactions (p < 0.10):")
print(int_df[int_df["p"] < 0.10].to_string(index=False))

---
## 6. Summary

In [ ]:
print("=" * 70)
print("POSTER FEATURES SUMMARY — USER CHARACTERISTICS AS MODERATORS")
print("=" * 70)

print("\n[Verified accounts]")
if "xpoz_verified" in df.columns:
    n_ver = df["xpoz_verified"].sum()
    print(f"  {n_ver} verified accounts in dataset ({100*n_ver/len(df):.1f}% of tweets)")

print("\n[Bot-like accounts]")
if "xpoz_inauthentic_prob" in df.columns:
    hi_bot = (df["xpoz_inauthentic_prob"] > 0.5).sum()
    print(f"  {hi_bot} tweets from high-bot-probability accounts (prob > 0.5)")

print("\n[Significant moderation effects (p < 0.05)]")
if len(int_df) > 0:
    sig = int_df[int_df["sig"] == "*"]
    if len(sig) == 0:
        print("  None found.")
    for _, r in sig.iterrows():
        dirn = "stronger suppression" if r["coef"] > 0 else "weaker suppression / backfire"
        print(f"  {r['metric']} × {r['feature']}: coef={r['coef']:+.2f} → {dirn}")

print(f"\n[Output files]")
print(f"  {OUT_DIR}")
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(".png"):
        print(f"  ├── {f}")